# Demo 2 — Text Attacks
## Synonym Substitution with TextAttack (TextFooler)

**Threat Model:** Black-box query-based

**Goal:** Flip a sentiment classifier's prediction by replacing words with
semantically identical synonyms — invisible to a human reader.

**Library:** `textattack` (wraps HuggingFace models)

In [ ]:
# Install dependencies
# !pip install textattack transformers datasets torch

In [ ]:
import textattack
from textattack.models.wrappers import HuggingFaceModelWrapper
from textattack.datasets import HuggingFaceDataset
from textattack.attack_recipes import TextFoolerJin2019
from textattack import Attacker, AttackArgs
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import warnings, importlib.metadata
import nltk

warnings.filterwarnings('ignore')

# Download required NLTK data for TextAttack (prevents LookupError)
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('universal_tagset')

print('TextAttack version:', importlib.metadata.version('textattack'))

## Step 1 — Load a Pre-trained Sentiment Model

We use `distilbert-base-uncased-finetuned-sst-2-english`, a lightweight model
fine-tuned on SST-2 that classifies sentences as **POSITIVE** or **NEGATIVE**.

In [ ]:
MODEL_NAME = 'distilbert-base-uncased-finetuned-sst-2-english'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
model_hf   = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model_wrapper = HuggingFaceModelWrapper(model_hf, tokenizer)

test_sentences = ['The movie was absolutely fantastic!', 'This film was terrible.']
results = model_wrapper(test_sentences)
labels  = ['NEGATIVE', 'POSITIVE']
for s, r in zip(test_sentences, results):
    pred = labels[r.argmax()]
    conf = r.max() * 100
    print(f'  Input      : {s}')
    print(f'  Prediction : {pred} ({conf:.1f}% confidence)\n')

## Step 2 — TextFooler Attack

### How TextFooler Works
1. **Rank words** by importance using deletion-based scoring
2. For each important word, find **synonym candidates** (WordNet / counter-fitted embeddings)
3. Replace with the synonym that most *increases* confidence for the **wrong** class
4. Stop when the prediction flips

Constraint: Replaced words must have cosine similarity > 0.5 in embedding space.

In [ ]:
dataset     = HuggingFaceDataset('sst2', split='validation')
attack      = TextFoolerJin2019.build(model_wrapper)
attack_args = AttackArgs(
    num_examples=5,          # Reduced from 20 to speed up execution
    query_budget=150,        # Limit queries per example to speed up execution
    log_to_csv='02_text_attack_results.csv',
    disable_stdout=True,
)
attacker       = Attacker(attack, dataset, attack_args)
attack_results = attacker.attack_dataset()

## Step 3 — Examine and Visualise Results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display
import difflib

df = pd.read_csv('02_text_attack_results.csv')
print('Attack Result Summary')
print(df['result_type'].value_counts().to_string())

successful = df[df['result_type'] == 'Successful']
print(f'\n{len(successful)}/{len(df)} attacks succeeded')

def color_diff(orig, pert):
    diff = list(difflib.ndiff(orig.split(), pert.split()))
    html = []
    for word in diff:
        if word.startswith('- '):
            html.append(f'<span style="color:red; text-decoration:line-through; font-weight:bold;">{word[2:]}</span>')
        elif word.startswith('+ '):
            html.append(f'<span style="color:green; font-weight:bold;">{word[2:]}</span>')
        elif word.startswith('  '):
            html.append(f'{word[2:]}')
    return ' '.join(html)

if len(successful) > 0:
    html_output = "<h3>Attack Examples (Differences Highlighted)</h3><ul>"
    for _, row in successful.head(5).iterrows():
        diff_html = color_diff(str(row['original_text']), str(row['perturbed_text']))
        html_output += f"<li style='margin-bottom:15px; font-size:16px;'>"
        html_output += f"<b>Original ({row['original_output']}):</b> {str(row['original_text'])}<br>"
        html_output += f"<b>Perturbed (<span style='color:red;'>{row['perturbed_output']}</span>):</b> {diff_html}"
        html_output += "</li>"
    html_output += "</ul>"
    display(HTML(html_output))

if len(successful) > 0:
    row = successful.iloc[0]
    import textwrap
    orig = textwrap.fill(str(row['original_text']), width=60)
    pert = textwrap.fill(str(row['perturbed_text']), width=60)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.axis('off')
    ax.text(0.05, 0.8, f'Original (Pred: {row["original_output"]})\n{orig}', fontsize=11, va='top', color='darkgreen', bbox=dict(facecolor='#e8f6e9', edgecolor='green', boxstyle='round,pad=0.5'))
    ax.text(0.05, 0.2, f'Perturbed (Pred: {row["perturbed_output"]})\n{pert}', fontsize=11, va='bottom', color='darkred', bbox=dict(facecolor='#fce4e4', edgecolor='red', boxstyle='round,pad=0.5'))
    plt.title('In Process: Text Attack (Synonym Substitution)', fontweight='bold')
    plt.tight_layout()
    plt.savefig('02_in_process_attack.png', dpi=150)
    plt.show()


In [ ]:
counts = df['result_type'].value_counts()
colors = {'Successful': '#e74c3c', 'Failed': '#27ae60', 'Skipped': '#95a5a6'}
bar_colors = [colors.get(c, '#bdc3c7') for c in counts.index]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(counts.index, counts.values, color=bar_colors, edgecolor='black')
ax.bar_label(bars, fmt='%d', padding=3, fontsize=12)
ax.set_title('After Attack: TextFooler Results on SST-2', fontsize=13, fontweight='bold')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('02_after_attack.png', dpi=150)
plt.show()

## Summary

TextFooler shows that **black-box text attacks require no gradient access**.
By exploiting semantic similarity constraints, attackers can craft adversarial text that:
- **Fools the model** (prediction flips)
- **Looks identical** to a human reader
- **Requires only query access**

**Defenses:** Adversarial training with augmented text, synonym-aware input filtering,
interval-bound propagation.